<a href="https://colab.research.google.com/github/naubergois/curso-yolo-colab/blob/main/notebooks/02-deteccao-em-video.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 2. Detecção quadro a quadro em vídeo

Vídeo é só uma sequência de fotos. Para detectar objetos nele, você lê um quadro, roda o modelo, desenha as caixas e grava o quadro num arquivo novo. Repete até o fim.

O exemplo é um clipe de cinco segundos (menos de 1 MB) em que a câmera passeia por uma sala com cadeiras, plantas, mesa e livros. Muita classe ao mesmo tempo, e roda rápido até em CPU.

> GPU deixa bem mais rápido: **Ambiente de execução → Alterar o tipo de ambiente → T4 GPU**.

In [ ]:
!pip install -q ultralytics

In [ ]:
import base64
import subprocess
import urllib.request
from pathlib import Path

from IPython.display import HTML, display


def baixar(url, destino):
    if not Path(destino).exists():
        urllib.request.urlretrieve(url, destino)
    print(f"{destino}: {Path(destino).stat().st_size / 1e6:.1f} MB")
    return destino


def mostrar_video(caminho, largura=720):
    """Converte para H.264 (o navegador não toca mp4v) e embute no notebook."""
    web = str(Path(caminho).with_suffix("")) + "_web.mp4"
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", caminho,
         "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart", web],
        check=True,
    )
    dados = base64.b64encode(Path(web).read_bytes()).decode()
    display(HTML(f'<video width="{largura}" controls autoplay muted loop>'
                 f'<source src="data:video/mp4;base64,{dados}" type="video/mp4"></video>'))
    return web


def baixar_para_maquina(caminho):
    try:
        from google.colab import files
        files.download(caminho)
    except ImportError:
        print(f"Fora do Colab: o arquivo está em {Path(caminho).resolve()}")

In [ ]:
from collections import Counter

import cv2
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
video = baixar(
    "https://github.com/ultralytics/assets/releases/download/v0.0.0/home-objects-compressed.mp4",
    "sala.mp4",
)

## Ler, detectar, gravar

O OpenCV lê o vídeo com `VideoCapture` e escreve com `VideoWriter`. A resolução e o FPS do arquivo de saída precisam bater com os do original, senão o vídeo sai corrompido.

In [ ]:
MAX_QUADROS = 150

cap = cv2.VideoCapture(video)
fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
altura = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"{largura}x{altura} a {fps} FPS")

out = cv2.VideoWriter("sala_detectada.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (largura, altura))
contagem = Counter()
quadros = 0

while quadros < MAX_QUADROS:
    ok, frame = cap.read()
    if not ok:
        break
    r = model(frame, verbose=False)[0]
    contagem.update(model.names[int(c)] for c in r.boxes.cls)
    out.write(r.plot())
    quadros += 1

cap.release()
out.release()
print(f"{quadros} quadros processados")
assert quadros > 0

## Assistir o resultado

In [ ]:
web = mostrar_video("sala_detectada.mp4")

## Quantas detecções de cada classe?

Repare que o número é de **detecções**, não de objetos. A mesma cadeira aparece em dezenas de quadros e é contada em cada um. Separar "a mesma cadeira" de "outra cadeira" é exatamente o que o rastreamento resolve na próxima aula.

In [ ]:
for classe, n in contagem.most_common():
    print(f"{classe:12s} {n:5d} detecções  (~{n / quadros:.1f} por quadro)")

## Baixar o vídeo

No Colab, a célula abaixo baixa o vídeo anotado para o seu computador.

In [ ]:
baixar_para_maquina(web)

## O que fica

Detecção em vídeo é um laço: ler, inferir, desenhar, gravar. O gargalo é a inferência, por isso GPU e modelos menores fazem tanta diferença. Para filtrar só uma classe, passe `classes=[0]` (pessoa) na chamada do modelo.